In [ ]:
%%bash
rm -rf ~/.cache/huggingface/hub
rm -rf ~/.cache/huggingface/transformers

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge
!pip install --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio
# !pip install torch torchvision torchaudio

In [ ]:
# !pip -q install -U peft
!pip -q install peft==0.20.0

In [ ]:
import redshift_connector

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import re
from io import StringIO
import os
import json
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
import yaml
import boto3
from tqdm.auto import tqdm

In [ ]:
from utils_follow_up.ask_sa_followup import build_followups, FollowupLLM

In [ ]:
with open("./variables.yaml", "r") as yamlfile:
    cfg = yaml.load(yamlfile, Loader=yaml.FullLoader)

REGION_NAME = cfg['REGION_NAME']
SECRET_NAME = cfg['SECRET_NAME']


BUCKET_NAME = cfg['BUCKET_NAME']
S3_PROJECT_DIR = cfg['S3_PROJECT_DIR']
UPDATE_TABLE_DIR = cfg['UPDATE_TABLE_DIR']

HOST = cfg['HOST']
PORT = cfg['PORT']
DATABASE = cfg['DATABASE']

os.environ["HF_TOKEN"] = cfg['HFT']

In [ ]:
session = boto3.session.Session()
secrets_manager_client = session.client(
    service_name = 'secretsmanager',
    region_name = REGION_NAME
)

In [ ]:
secret_name = SECRET_NAME
response = secrets_manager_client.get_secret_value(SecretId=secret_name)

if 'SecretString' in response:
    secret = response['SecretString']
else:
    secret = response['SecretBinary']

In [ ]:
parsed_secret = json.loads(secret)
redshift_username = parsed_secret['username']
redshift_password = parsed_secret['password']

In [ ]:
obj = boto3.client('s3')

In [ ]:
querie_get_data = './queries/get_chats.sql'
fd = open(querie_get_data, 'r')
get_data = fd.read()
fd.close()

In [ ]:
conn = redshift_connector.connect(
    host = cfg['HOST'],
    port = cfg['PORT'],
    database = cfg['DATABASE'],
    user = redshift_username,
    password = redshift_password
 )
cur = conn.cursor()

In [ ]:
cur.execute(get_data)
chat_data_df = cur.fetch_dataframe()
chat_data_df.head()

In [ ]:
followups = build_followups(chat_data_df)
print("Follow-ups extracted:", len(followups))
followups.head()

In [ ]:
llm = FollowupLLM(
    # model_id="microsoft/Phi-3.5-mini-instruct",
    # model_id="Qwen/Qwen2.5-7B-Instruct",
    model_id="google/gemma-2-9b-it",
    max_input_tokens=1400,
    max_new_tokens=96,
    batch_size=2,
    token=os.getenv("HF_TOKEN"),
    enable_json_repair=True,
)

In [ ]:
scored = llm.classify_df(followups)
scored.head(10)

In [ ]:
print(scored["label"].value_counts(dropna=False))
print("\nTop subtypes:")
print(scored["subtype"].value_counts(dropna=False).head(20))

print("\nRows that were likely parse-related:")
display(scored[scored["rationale"].isin(["no_json","json_parse_error"])][
    ["followup_query","raw_model_text","label","subtype","rationale"]
].head(10))

In [ ]:
conn_info = {'host': cfg['HOST'],
             'port': cfg['PORT'],
             'user': redshift_username,
             'password': redshift_password,
             'database': cfg['DATABASE']}

In [ ]:
# with redshift_connector.connect(**conn_info) as conn:
#     cur = conn.cursor()

#     drop_query = 'DROP TABLE IF EXISTS ds.production__ask_sa_sentiment_followup_msg_analysis'
#     cur.execute(drop_query)

#     create_table = 'CREATE TABLE ds.production__ask_sa_sentiment_followup_msg_analysis (user_id int, chat_id int, followup_number int,\
#                                                                                          followup_row_index int, followup_time timestamp,\
#                                                                                          initial_query varchar(MAX), ask_sa_answer varchar(MAX),\
#                                                                                          followup_query varchar(MAX), label varchar(20),\
#                                                                                          subtype varchar(40), confidence float,\
#                                                                                          rationale varchar(MAX), model_used varchar(50),\
#                                                                                          raw_model_text varchar(MAX), followup_date date) DISTSTYLE EVEN SORTKEY(followup_date)'
#     cur.execute(create_table)

#     access = 'GRANT ALL PRIVILEGES ON TABLE ds.production__ask_sa_sentiment_followup_msg_analysis TO analyst'

#     cur.execute(access)

#     access = 'GRANT ALL PRIVILEGES ON TABLE ds.production__ask_sa_sentiment_followup_msg_analysis TO scientist'

#     cur.execute(access)
    
#     access = 'GRANT ALL PRIVILEGES ON TABLE ds.production__ask_sa_sentiment_followup_msg_analysis TO looker'
    
#     cur.execute(access)
    
#     conn.commit()

In [ ]:
scored['followup_date'] = scored['followup_time'].dt.date
scored[['user_id', 'chat_id']] = scored[['user_id', 'chat_id']].astype(int)

In [ ]:
# Save  the first 256 characters
cols = ['initial_query', 'ask_sa_answer', 'followup_query', 'rationale', 'raw_model_text']
scored[cols] = scored[cols].apply(lambda s: s.astype("string").str.slice(0, 256))

In [ ]:
csv_buf = StringIO()
scored.to_csv(csv_buf, header=True, index=False)
csv_buf.seek(0)
obj.put_object(Bucket = BUCKET_NAME, Body=csv_buf.getvalue(), Key = os.path.join(S3_PROJECT_DIR + UPDATE_TABLE_DIR, 'result.csv'))

In [ ]:
with redshift_connector.connect(**conn_info) as conn:
     cur = conn.cursor()
     file_path = 's3://' +  BUCKET_NAME + '/' + S3_PROJECT_DIR + UPDATE_TABLE_DIR + 'result.csv'
     role_arn = 'arn:aws:iam::714212176917:role/redshift-access-to-seekingalpha-datascience-bucket'

     insert_query = """COPY ds.production__ask_sa_sentiment_followup_msg_analysis FROM '{}' IAM_ROLE '{}' csv IGNOREHEADER 1;""".format(file_path, role_arn)
     cur.execute(insert_query)
     conn.commit()

In [ ]:
with redshift_connector.connect(**conn_info) as conn:
    cur = conn.cursor()

    analyze_query = 'ANALYZE ds.production__ask_sa_sentiment_followup_msg_analysis'
    cur.execute(analyze_query)
    
    conn.commit()